# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lokeshtiwari723/Proto-ex/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I will use Logistic Regression as the first model for this ranking task. It is simple, interpretable, and gives a transparent baseline for checking whether the selected February signals contain useful information about the March outcome. I will compare its results with my Week-4 action-score baseline using the same data split and evaluation metric. The goal is decision support for prioritizing pages for review, not a final causal judgment.

In [12]:
import os
import duckdb
import pandas as pd

print("Imports OK")
print("HF_TOKEN available:", bool(os.environ.get("HF_TOKEN")))

Imports OK
HF_TOKEN available: False


In [13]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", bool(HF_TOKEN))

HF_TOKEN loaded: True


In [14]:
# Connect to the FlyRank warehouse

from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

print("HF token accepted by Hugging Face API")

HF token accepted by Hugging Face API


In [15]:
# Connect DuckDB to the FlyRank Hugging Face warehouse

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

DIM = f"read_parquet('{REL}/dim_content.parquet')"

print("Warehouse connection ready")
print("Feature window: February 2026")
print("Label window: March 2026")

Warehouse connection ready
Feature window: February 2026
Label window: March 2026


In [16]:
features = con.sql(f"""
WITH feb_page AS (
    SELECT
        client_hash_id,
        content_hash_id,

        -- February monthly totals
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,

        -- Reconstruct February average position at page level
        SUM(gsc_sum_position)
            / NULLIF(SUM(gsc_impressions), 0)
            AS gsc_avg_position

    FROM {FEB}
    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.gsc_impressions,
    f.gsc_clicks,
    f.gsc_avg_position,
    d.word_count,
    d.content_created_date

FROM feb_page AS f

LEFT JOIN {DIM} AS d
    ON f.client_hash_id = d.client_hash_id
    AND f.content_hash_id = d.content_hash_id

""").df()

print("Page-level February rows:", len(features))

print(
    "Unique client-content pairs:",
    features[
        ["client_hash_id", "content_hash_id"]
    ].drop_duplicates().shape[0]
)

duplicate_count = (
    features
    .duplicated(
        subset=["client_hash_id", "content_hash_id"]
    )
    .sum()
)

print("Duplicate page records:", duplicate_count)

display(features.head())

Page-level February rows: 153559
Unique client-content pairs: 153559
Duplicate page records: 0


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,word_count,content_created_date
0,client_3ffa76342f366962,content_36bee0a093d0711d,3.0,0.0,7.000000,815,2025-09-06
1,client_3ffa76342f366962,content_1546aabff77c05a4,5.0,1.0,4.200000,957,2025-09-07
2,client_3ffa76342f366962,content_cae1d5374958a649,96.0,0.0,5.520833,861,2025-09-08
3,client_3ffa76342f366962,content_dd66eecf9626cab8,235.0,0.0,6.391489,823,2025-09-08
4,client_3ffa76342f366962,content_c51f1e8ef5502177,18.0,0.0,7.000000,786,2025-09-26


## 2. Split design

I will use a grouped train/test split by client_hash_id. This keeps pages from the same client from appearing in both training and test sets, reducing the risk that the model learns client-specific patterns instead of generalizable page-level signals. The client ID is used only for grouping, not as a model feature. I will use the same held-out test set to compare Logistic Regression with the Week-4 baseline using Precision@K.

In [17]:
# ML-08 — Build the same February feature + March label frame as ML-07

# Reuse the warehouse connection and constants from the earlier ML-04/ML-07 setup.
# HF_TOKEN should already be stored in Colab Secrets.

# Build the March outcome label

label_df = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_clicks) AS march_clicks,
    SUM(gsc_impressions) AS march_impressions

FROM {MAR}

WHERE gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id

HAVING SUM(gsc_impressions) >= 100
""").df()

label_df["march_ctr"] = (
    label_df["march_clicks"]
    / label_df["march_impressions"]
)

ctr_cutoff = label_df["march_ctr"].median()

label_df["label"] = (
    label_df["march_ctr"] > ctr_cutoff
).astype(int)

print("March label rows:", len(label_df))
print("March CTR median cutoff:", round(ctr_cutoff, 6))

print("\nLabel distribution:")
print(label_df["label"].value_counts().sort_index())

March label rows: 101441
March CTR median cutoff: 0.001238

Label distribution:
label
0    50722
1    50719
Name: count, dtype: int64


In [18]:
# Join February features to the March outcome

audit_df = features.merge(
    label_df[
        [
            "client_hash_id",
            "content_hash_id",
            "march_ctr",
            "label"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Convert content date and calculate age as of the end of February

audit_df["content_created_date"] = pd.to_datetime(
    audit_df["content_created_date"],
    errors="coerce"
)

decision_date = pd.Timestamp("2026-02-28")

audit_df["content_age_days"] = (
    decision_date
    - audit_df["content_created_date"]
).dt.days

# Keep valid records for modeling

model_df = audit_df.dropna(
    subset=[
        "label",
        "gsc_impressions",
        "gsc_avg_position",
        "content_age_days"
    ]
).copy()

# Remove impossible future-created content records

model_df = model_df[
    model_df["content_age_days"] >= 0
].copy()

print("Rows after February/March join:", len(audit_df))
print("Rows available for modeling:", len(model_df))

print(
    "Unique client-content pairs:",
    model_df[
        ["client_hash_id", "content_hash_id"]
    ].drop_duplicates().shape[0]
)

print(
    "Duplicate page records:",
    model_df.duplicated(
        subset=["client_hash_id", "content_hash_id"]
    ).sum()
)

display(
    model_df[
        [
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position",
            "word_count",
            "content_age_days",
            "label"
        ]
    ].head()
)

Rows after February/March join: 86560
Rows available for modeling: 86560
Unique client-content pairs: 86560
Duplicate page records: 0


,gsc_impressions,gsc_clicks,gsc_avg_position,word_count,content_age_days,label
0,235.0,0.0,6.391489,823,173,0
1,660.0,0.0,61.680303,<NA>,344,0
2,1139.0,0.0,49.698859,<NA>,344,0
3,1063.0,0.0,54.554092,<NA>,344,0
4,586.0,0.0,42.027304,3000,344,0


In [19]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        model_df,
        model_df["label"],
        groups=model_df["client_hash_id"]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

train_clients = set(train_df["client_hash_id"])
test_clients = set(test_df["client_hash_id"])

client_overlap = len(
    train_clients.intersection(test_clients)
)

print("Total modeling rows:", len(model_df))
print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))

print("Client overlap:", client_overlap)

print(
    "Train positive rate:",
    round(train_df["label"].mean(), 4)
)

print(
    "Test positive rate:",
    round(test_df["label"].mean(), 4)
)

Total modeling rows: 86560
Train rows: 75670
Test rows: 10890
Train clients: 29
Test clients: 8
Client overlap: 0
Train positive rate: 0.481
Test positive rate: 0.5296


## 3. Train + compare vs my baseline

I compare Logistic Regression with the Week-4 action-score baseline on the same held-out test rows.

The model uses only February information available before the March outcome window:

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- word_count
- content_age_days

Identifiers are not model features. March CTR, March clicks, March impressions, the binary label, trend_direction, and trend_pct are excluded from the feature set.

The main ranking metric is Precision@50 because the task is to prioritize a small review queue rather than make a final classification decision for every page.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "word_count",
    "content_age_days"
]

X_train = train_df[feature_cols]
y_train = train_df["label"]

X_test = test_df[feature_cols]
y_test = test_df["label"]

model = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "logreg",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

model.fit(X_train, y_train)

model_scores = model.predict_proba(
    X_test
)[:, 1]

print("Model trained successfully")
print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Features:", feature_cols)

Model trained successfully
Train rows: 75670
Test rows: 10890
Features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'word_count', 'content_age_days']


In [21]:
# Re-create the Week-4 baseline rule on exactly the same test rows

test_eval = test_df.copy()

visible = (
    test_eval["gsc_impressions"] >= 100
)

stale = (
    test_eval["content_age_days"] >= 180
)

test_eval["baseline_score"] = (
    visible.astype(int)
    + stale.astype(int)
)

test_eval["model_score"] = model_scores

print("Baseline score distribution:")
print(
    test_eval["baseline_score"]
    .value_counts()
    .sort_index()
)

Baseline score distribution:
baseline_score
0    2080
1    5392
2    3418
Name: count, dtype: int64


In [22]:
K = 50

# Logistic Regression ranking
model_top_k = (
    test_eval
    .sort_values(
        "model_score",
        ascending=False
    )
    .head(K)
)

model_precision_at_k = (
    model_top_k["label"].mean()
)

# Week-4 baseline ranking
# Same tie-break logic as baseline work:
# score first, then February impressions

baseline_top_k = (
    test_eval
    .sort_values(
        ["baseline_score", "gsc_impressions"],
        ascending=[False, False]
    )
    .head(K)
)

baseline_precision_at_k = (
    baseline_top_k["label"].mean()
)

comparison = pd.DataFrame(
    {
        "Method": [
            "Week-4 action-score baseline",
            "Logistic Regression"
        ],
        "Precision@50": [
            baseline_precision_at_k,
            model_precision_at_k
        ],
        "Positive pages in Top 50": [
            int(baseline_top_k["label"].sum()),
            int(model_top_k["label"].sum())
        ]
    }
)

print("Model vs baseline comparison:")
display(comparison)

Model vs baseline comparison:


,Method,Precision@50,Positive pages in Top 50
0,Week-4 action-score baseline,0.76,38
1,Logistic Regression,1.00,50


In [23]:
coefficients = pd.DataFrame(
    {
        "feature": feature_cols,
        "coefficient": (
            model.named_steps["logreg"]
            .coef_[0]
        )
    }
)

coefficients["absolute_coefficient"] = (
    coefficients["coefficient"].abs()
)

coefficients = coefficients.sort_values(
    "absolute_coefficient",
    ascending=False
)

print(
    "Standardized Logistic Regression coefficients:"
)

display(coefficients)

Standardized Logistic Regression coefficients:


,feature,coefficient,absolute_coefficient
1,gsc_clicks,9.709070,9.709070
0,gsc_impressions,-2.122548,2.122548
2,gsc_avg_position,-0.494703,0.494703
3,word_count,-0.075091,0.075091
4,content_age_days,0.021170,0.021170


## 4. Errors and interpretation

IThe Logistic Regression model placed 50 positive pages in its Top 50 on the held-out test set, giving a measured Precision@50 of 1.00. The Week-4 action-score baseline had a Precision@50 of 0.76 with 38 positive pages in its Top 50.

In this test split, the model had 0 false positives among its Top 50. However, 5,717 positive pages were ranked outside the Top 50, so a perfect Precision@50 does not mean the model identifies every positive page.

The model coefficients show the strongest absolute standardized coefficient for gsc_clicks, followed by gsc_impressions and gsc_avg_position. These are model associations within this fitted model, not causal effects.

The result supports using the model as a narrow review-prioritization aid on this held-out split. It should not be treated as a final verdict about page quality or search performance.

Human review is still required because the proxy label is based on the March CTR threshold and the available February signals do not capture every factor that may affect later performance.

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Inspect errors in the model's Top-50 ranking

model_ranked = (
    test_eval
    .sort_values(
        "model_score",
        ascending=False
    )
    .reset_index(drop=True)
)

model_ranked["model_rank"] = (
    np.arange(1, len(model_ranked) + 1)
)

top50 = model_ranked.head(50).copy()

top50_false_positives = (
    top50[top50["label"] == 0].copy()
)

missed_positives = (
    model_ranked[
        (model_ranked["model_rank"] > 50)
        & (model_ranked["label"] == 1)
    ]
    .head(10)
    .copy()
)

print(
    "Top-50 correct positives:",
    int((top50["label"] == 1).sum())
)

print(
    "Top-50 false positives:",
    int((top50["label"] == 0).sum())
)

print(
    "Positive pages outside Top 50:",
    int(
        (
            (model_ranked["model_rank"] > 50)
            & (model_ranked["label"] == 1)
        ).sum()
    )
)

print("\nExample Top-50 false positives:")

display(
    top50_false_positives[
        [
            "model_rank",
            "model_score",
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position",
            "word_count",
            "content_age_days",
            "label"
        ]
    ].head(10)
)

print("\nExample positive pages missed by Top 50:")

display(
    missed_positives[
        [
            "model_rank",
            "model_score",
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position",
            "word_count",
            "content_age_days",
            "label"
        ]
    ]
)

Top-50 correct positives: 50
Top-50 false positives: 0
Positive pages outside Top 50: 5717

Example Top-50 false positives:


,model_rank,model_score,gsc_impressions,gsc_clicks,gsc_avg_position,word_count,content_age_days,label



Example positive pages missed by Top 50:


,model_rank,model_score,gsc_impressions,gsc_clicks,gsc_avg_position,word_count,content_age_days,label
50,51,1.0,4386.0,64.0,2.402873,1150,359,1
51,52,1.0,5409.0,66.0,10.322980,2781,162,1
52,53,1.0,14712.0,76.0,7.892537,2634,232,1
53,54,1.0,3047.0,61.0,4.110929,1608,209,1
54,55,1.0,10601.0,71.0,8.751344,3244,359,1
55,56,1.0,1419.0,58.0,2.392530,1120,232,1
56,57,1.0,6747.0,64.0,3.994220,2834,183,1
57,58,1.0,7399.0,65.0,6.715367,2843,162,1
58,59,1.0,7146.0,65.0,16.456759,2309,64,1
59,60,1.0,16014.0,74.0,1.985450,2842,64,1


In [25]:
# Final leakage and validation checks

prohibited_features = {
    "client_hash_id",
    "content_hash_id",
    "march_clicks",
    "march_impressions",
    "march_ctr",
    "label",
    "trend_direction",
    "trend_pct"
}

unexpected_features = (
    set(feature_cols)
    & prohibited_features
)

print("Model features:", feature_cols)
print("Prohibited fields used as features:", unexpected_features)

print(
    "Client overlap:",
    len(
        set(train_df["client_hash_id"])
        & set(test_df["client_hash_id"])
    )
)

print(
    "Duplicate modeling records:",
    model_df.duplicated(
        subset=[
            "client_hash_id",
            "content_hash_id"
        ]
    ).sum()
)

if (
    len(unexpected_features) == 0
    and client_overlap == 0
):
    print("Leakage / split check: PASSED")
else:
    print("Leakage / split check: REVIEW REQUIRED")

Model features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'word_count', 'content_age_days']
Prohibited fields used as features: set()
Client overlap: 0
Duplicate modeling records: 0
Leakage / split check: PASSED


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.